In [ ]:
from dmipy.signal_models import sphere_models, cylinder_models, gaussian_models

In [ ]:
import numpy as np  
from dmipy.core.acquisition_scheme import acquisition_scheme_from_bvalues

In [ ]:
bvalues = np.loadtxt('../data/grad_files/bvals_verdict.txt')  # given in s/mm^2
bvalues_SI = bvalues * 1e6  # now given in SI units as s/m^2
gradient_directions = np.loadtxt('../data/grad_files/bvecs_verdict.txt')  # on the unit sphere

# The delta and Delta times we know from the HCP documentation in seconds
delta = np.loadtxt('../data/grad_files/smalldelta_verdict.txt') * 1e-3  # in seconds 
Delta = np.loadtxt('../data/grad_files/delta_verdict.txt') * 1e-3  # in seconds

# The acquisition scheme we use in the toolbox is then created as follows:
acq_scheme = acquisition_scheme_from_bvalues(bvalues_SI, gradient_directions.T, delta, Delta)

In [ ]:
sphere = sphere_models.S4SphereGaussianPhaseApproximation(diffusion_constant=2e-9)
ball = gaussian_models.G1Ball()
stick = cylinder_models.C1Stick()

In [ ]:
from dmipy.core.modeling_framework import MultiCompartmentSphericalMeanModel
verdict_mod = MultiCompartmentSphericalMeanModel(models=[sphere, ball, stick])
verdict_mod.set_fixed_parameter('G1Ball_1_lambda_iso', 2e-9)
verdict_mod.set_fixed_parameter('C1Stick_1_lambda_par', 8e-9)
verdict_mod.parameter_names

In [ ]:
data_verdict_path = '../data/test_images/BallSphereAstrosticks_fixed.nii.gz'
mask_verdict_path = '../data/test_images/BallSphereAstrosticks_fixed_mask.nii.gz'

In [ ]:
import nibabel as nib
data_verdict = nib.load(data_verdict_path).get_fdata()
mask_verdict = nib.load(mask_verdict_path).get_fdata()

In [ ]:
verdict_fit = verdict_mod.fit(acq_scheme, data=data_verdict, mask=mask_verdict)